# Period Finding & Shor — IQM edition

This notebook runs on IQM **Garnet** (20 qubits) or **Emerald** (54 qubits) via IQM Resonance,
transpiling to the native IQM gate set **PRX + CZ**. Pick the backend by editing the single
`TARGET` line below (see `README.md` and `iqm_backend.py` in this folder):

| `TARGET` | runs on | token? | QPU credits? |
|---|---|---|---|
| `"fake_apollo"` | local 20-qubit noisy simulator | no | no |
| `"garnet:mock"` / `"emerald:mock"` | IQM Resonance cloud mock (pipeline check: returns RANDOM bitstrings, no physics) | yes | no |
| `"garnet"` / `"emerald"` | **real QPU** (20q / 54q) | yes | yes |

For the full, enriched treatment (classical reductions, robust `shor_factor`, breaking a toy RSA
key) see `shor.ipynb` in this folder.

In [ ]:
from iqm_backend import get_backend, run_counts   # helper module in this folder (see iqm_backend.py)
# run_counts(circuit, TARGET, shots=...) = transpile to the IQM native gates (PRX + CZ) + run + get_counts()
# get_backend(TARGET) returns the raw Qiskit backend object if you want to call backend.run yourself
TARGET = "fake_apollo"   # change this one line to choose the backend (see the table above)


In [ ]:
import matplotlib.pyplot as plt        # plotting
import numpy as np                     # numerical arrays and np.pi
from qiskit import QuantumCircuit                 # the object we build circuits with
from qiskit.visualization import plot_histogram   # bar chart of measurement counts
import math
from math import gcd                   # greatest common divisor
from numpy.random import randint       # random integers
import pandas as pd                    # tables (DataFrame) for printing results
from fractions import Fraction         # exact rational numbers; .limit_denominator() = continued fractions
print("Imports Successful")


In [ ]:
# Define the number to be factored (N)
N = 15

# Define the number of qubits needed to store the number N
n = math.ceil(math.log(N, 2))

# Define the number of qubits needed to store the value of a
m = 2 * n
a = 3

In [ ]:
print(N,n,m,a)

In [ ]:
# Calculate the plotting data
xvals = np.arange(N)                         # NumPy array 0, 1, ..., N-1
yvals = [np.mod(a**x, N) for x in xvals]     # list comprehension: a^x mod N for every x

# Use matplotlib to display it nicely
fig, ax = plt.subplots()                     # one figure with one set of axes
ax.plot(xvals, yvals, linewidth=1, linestyle='dotted', marker='x')
ax.set(xlabel='$x$', ylabel=f'${a}^x$ mod ${N}$',          # f-string: {a} and {N} are replaced by their values
       title="Example of Periodic Function in Shor's Algorithm")
try: # plot r on the graph
    r = yvals[1:].index(1) + 1               # first x > 0 with a^x mod N == 1, i.e. the period r
    plt.annotate('', xy=(0,1), xytext=(r,1),
                 arrowprops=dict(arrowstyle='<->'))
    plt.annotate(f'$r={r}$', xy=(r/3,1.5))
except ValueError:                           # list.index() raises ValueError if 1 never appears
    print('Could not find period, check a < N and have no common factors.')


## Implementation of period finding
In this example we will solve the period finding problem for $a=7$ and $N=15$
. We provide the circuits for 
 where:
$U|y\rangle = |ay\bmod 15\rangle$ without explanation. To create $U^x$ , we will simply repeat the circuit $x$ 
 times. In the next section we will discuss a general method for creating these circuits efficiently. The function c_amod15 returns the controlled-U gate for a, repeated power times.

In [ ]:
# Specify variables
N_COUNT = 8  # number of counting qubits
a = 7

In [ ]:
def c_amod15(a, power):
    """Controlled multiplication by a^power mod 15.

    IQM note: a^power mod 15 is computed CLASSICALLY (repeated squaring) and the circuit
    does a single multiplication by it, instead of repeating U `power` times. The naive
    loop produces ~70 000 native instructions after transpilation (IQM Resonance refuses
    circuits above 10 000); this version needs ~650."""
    if a not in [2,4,7,8,11,13]:
        raise ValueError("'a' must be 2,4,7,8,11 or 13")   # raise = abort with an error message
    a_eff = pow(a, power, 15)   # pow(a, k, N) = a^k mod N; one of 1,2,4,7,8,11,13 ; a_eff == 1 -> identity
    U = QuantumCircuit(4)       # a 4-qubit circuit (no classical bits): the work register
    # U.swap(i, j) exchanges qubits i and j; U.x(q) applies a NOT (Pauli X) to qubit q.
    if a_eff in [2,13]:
        U.swap(2,3)
        U.swap(1,2)
        U.swap(0,1)
    if a_eff in [7,8]:
        U.swap(0,1)
        U.swap(1,2)
        U.swap(2,3)
    if a_eff in [4, 11]:
        U.swap(1,3)
        U.swap(0,2)
    if a_eff in [7,11,13]:
        for q in range(4):
            U.x(q)
    U = U.to_gate()                       # pack the circuit into a single reusable Gate object
    U.name = f"{a}^{power} mod 15"        # label shown in circuit drawings
    c_U = U.control()                     # add one control qubit -> controlled-U
    return c_U


In [ ]:
def qft_dagger(n):
    """n-qubit QFTdagger the first n qubits in circ"""
    qc = QuantumCircuit(n)
    # Don't forget the Swaps!
    for qubit in range(n//2):                        # reverse the qubit order
        qc.swap(qubit, n-qubit-1)
    for j in range(n):
        for m in range(j):
            qc.cp(-np.pi/float(2**(j-m)), m, j)      # qc.cp(theta, control, target): controlled phase gate
        qc.h(j)                                      # Hadamard on qubit j
    qc.name = "QFT†"                                 # label shown in circuit drawings
    return qc


Let us look at the two building blocks before assembling them. `draw('mpl')` renders a circuit with
matplotlib. A gate has no `draw()`, so for the controlled multiplication we draw `base_gate.definition`
(the circuit of $U$ without its control); `decompose()` expands composite gates by one level.


In [ ]:
# U for a = 7, power = 1: multiplication by 7 mod 15 on the 4-qubit work register
display(c_amod15(7, 1).base_gate.definition.draw('mpl'))   # display(): show a figure that is not the last line

# The controlled version inside a 5-qubit circuit (qubit 0 = control), expanded one level
cU = QuantumCircuit(5)
cU.append(c_amod15(7, 1), [0, 1, 2, 3, 4])    # append(gate, qubits): control first, then the 4 work qubits
display(cU.decompose().draw('mpl'))

# The inverse QFT on 4 qubits (the last expression of a cell is displayed automatically)
qft_dagger(4).draw('mpl')


In [ ]:
# Create QuantumCircuit with N_COUNT counting qubits
# plus 4 qubits for U to act on
qc = QuantumCircuit(N_COUNT + 4, N_COUNT)   # QuantumCircuit(number_of_qubits, number_of_classical_bits)

# Initialize counting qubits
# in state |+>
for q in range(N_COUNT):
    qc.h(q)                                 # Hadamard on qubit q

# And auxiliary register in state |1>
qc.x(N_COUNT)                               # NOT on the first work qubit

# Do controlled-U operations
for q in range(N_COUNT):
    # qc.append(gate, qubits): the list names the qubits the gate acts on, control first
    qc.append(c_amod15(a, 2**q),
             [q] + [i+N_COUNT for i in range(4)])

# Do inverse-QFT
qc.append(qft_dagger(N_COUNT), range(N_COUNT))

# Measure circuit
qc.measure(range(N_COUNT), range(N_COUNT))  # measure(qubits, classical_bits): qubit q -> bit q
print(f"{qc.num_qubits} qubits, {qc.num_clbits} classical bits, depth {qc.depth()}")


In [ ]:
qc.draw('mpl',fold=-1)  # -1 means 'do not fold'

In [ ]:
# run_counts = transpile_to_IQM (native PRX + CZ gates) + backend.run + get_counts.
# return_transpiled=True also returns the transpiled circuit, so we can see what the hardware executes.
counts, qc_iqm = run_counts(qc, TARGET, shots=2048, return_transpiled=True)
print(f"transpiled for IQM: {qc_iqm.size()} native instructions, depth {qc_iqm.depth()}, "
      f"gate counts {dict(qc_iqm.count_ops())}")   # 'r' = PRX rotation, 'cz' = controlled-Z
plot_histogram(counts)                              # counts = dict {'bitstring': how many shots gave it}


In [ ]:
rows, measured_phases = [], []
for output in counts:                 # iterating over a dict gives its keys: the measured bitstrings
    decimal = int(output, 2)  # Convert (base 2) string to decimal
    phase = decimal/(2**N_COUNT)  # Find corresponding eigenvalue
    measured_phases.append(phase)
    # Add these values to the rows in our table:
    rows.append([f"{output}(bin) = {decimal:>3}(dec)",     # :>3 = right-align in 3 characters
                 f"{decimal}/{2**N_COUNT} = {phase:.2f}"])   # :.2f = 2 decimal places
# Print the rows in a table
headers=["Register Output", "Phase"]
df = pd.DataFrame(rows, columns=headers)       # pandas table
print(df)


In [ ]:
rows = []
for phase in measured_phases:
    frac = Fraction(phase).limit_denominator(15)   # continued fractions: closest s/r with r <= 15
    rows.append([phase,
                 f"{frac.numerator}/{frac.denominator}",
                 frac.denominator])
# Print as a table
headers=["Phase", "Fraction", "Guess for r"]
df = pd.DataFrame(rows, columns=headers)
print(df)


We can see that two of the measured eigenvalues provided us with the correct result: $r=4$, and we can see that Shor’s algorithm has a chance of failing. 

In [ ]:
r = 4

In [ ]:
# Use the period r to calculate the factors of N using classical algorithms
gcd = math.gcd(a**(r//2)-1, N)   # // = integer division; math.gcd = Euclid's algorithm
factor1 = gcd
factor2 = N // gcd


In [ ]:
# Print the factors of N
print("The factors of", N, "are", factor1, "and", factor2)